In [1]:
import os, json
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import torch.nn as nn
import copy
from torch_scatter import scatter_mean
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GCNConv
from torch.optim import Adam
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, accuracy_score, classification_report
from tqdm import tqdm

BIGCN_DIR = "/teamspace/studios/this_studio/BiGCN"
OUTPUTS   = "/teamspace/studios/this_studio/misinformation-detection/outputs/twitter_tfidf"
os.makedirs(OUTPUTS, exist_ok=True)

device   = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
LABEL_MAP = {'false': 0, 'true': 1, 'unverified': 2, 'non-rumor': 3}
print(f"Device: {device}")

Device: cuda


In [2]:
def load_labels_bigcn(path):
    """Load from BiGCN repo label format: label\t...\ttweet_id\t..."""
    labels = {}
    with open(path) as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            parts = line.split('\t')
            label    = parts[0].lower()
            tweet_id = parts[2]
            if label in ['news', 'non-rumor']:
                labels[tweet_id] = LABEL_MAP['non-rumor']
            elif label in LABEL_MAP:
                labels[tweet_id] = LABEL_MAP[label]
    return labels

t15_labels = load_labels_bigcn(
    f'{BIGCN_DIR}/data/Twitter15/Twitter15_label_All.txt')
t16_labels = load_labels_bigcn(
    f'{BIGCN_DIR}/data/Twitter16/Twitter16_label_All.txt')

print(f"T15 labels: {len(t15_labels)}")
print(f"T16 labels: {len(t16_labels)}")

# check class distribution
from collections import Counter
inv_map = {v: k for k, v in LABEL_MAP.items()}
print("\nT15:", {inv_map[k]: v for k, v in
                Counter(t15_labels.values()).items()})
print("T16:", {inv_map[k]: v for k, v in
                Counter(t16_labels.values()).items()})

T15 labels: 1490
T16 labels: 818

T15: {'unverified': 374, 'non-rumor': 374, 'true': 372, 'false': 370}
T16: {'false': 205, 'true': 207, 'unverified': 201, 'non-rumor': 205}


In [3]:
def parse_tfidf_line(feat_str, vocab_size=5000):
    """Parse sparse TF-IDF string into dense vector."""
    vec = np.zeros(vocab_size, dtype=np.float32)
    if feat_str.strip():
        for item in feat_str.strip().split():
            idx, val = item.split(':')
            vec[int(idx)] = float(val)
    return vec

def load_tfidf_dataset(data_path, labels_dict):
    """
    Parse data.TD_RvNN.vol_5000.txt into PyG Data objects.
    Format per line: root_id  parent_id  node_id  max_seq  total_words  feats
    """
    # first pass — group lines by root_id
    cascades = {}
    with open(data_path) as f:
        for line in f:
            line  = line.strip()
            if not line:
                continue
            parts = line.split('\t')
            if len(parts) < 6:
                continue

            root_id   = parts[0].strip()
            parent_id = parts[1].strip()
            node_id   = int(parts[2].strip())
            feat_str  = '\t'.join(parts[5:])

            if root_id not in cascades:
                cascades[root_id] = {'nodes': {}, 'edges': []}

            feat_vec = parse_tfidf_line(feat_str)
            cascades[root_id]['nodes'][node_id] = feat_vec

            if parent_id != 'None':
                cascades[root_id]['edges'].append(
                    (int(parent_id), node_id))

    # second pass — build PyG Data objects
    data_list  = []
    labels_out = []
    skipped    = 0

    for root_id, cascade in tqdm(cascades.items(),
                                  desc="building graphs"):
        if root_id not in labels_dict:
            skipped += 1
            continue

        label   = labels_dict[root_id]
        nodes   = cascade['nodes']
        edges   = cascade['edges']

        if len(nodes) < 2 or len(edges) == 0:
            skipped += 1
            continue

        # node index mapping (local ids may not be contiguous)
        node_list   = sorted(nodes.keys())
        node_to_idx = {n: i for i, n in enumerate(node_list)}
        n_nodes     = len(node_list)

        # feature matrix
        x = torch.tensor(
            np.stack([nodes[n] for n in node_list]),
            dtype=torch.float)                          # (n_nodes, 5000)

        # find root node (parent_id == None → that node_id is root)
        # root is the node that never appears as a child
        child_ids = {c for _, c in edges}
        root_candidates = [n for n in node_list if n not in child_ids]
        root_local = root_candidates[0] if root_candidates else node_list[0]
        root_idx   = node_to_idx[root_local]

        # TD edges
        td_edges = [(node_to_idx[p], node_to_idx[c])
                    for p, c in edges
                    if p in node_to_idx and c in node_to_idx]
        if len(td_edges) == 0:
            skipped += 1
            continue

        td_src, td_dst = zip(*td_edges)
        td_edge_index  = torch.tensor([td_src, td_dst], dtype=torch.long)
        bu_edge_index  = torch.tensor([td_dst, td_src], dtype=torch.long)

        data_list.append(Data(
            x             = x,
            edge_index    = td_edge_index,
            BU_edge_index = bu_edge_index,
            y             = torch.tensor([label], dtype=torch.long),
            root_index    = torch.tensor([root_idx], dtype=torch.long),
            num_nodes     = n_nodes
        ))
        labels_out.append(label)

    print(f"Built {len(data_list)} graphs, skipped {skipped}")
    return data_list, np.array(labels_out)

print("Loading Twitter15...")
t15_data, t15_y = load_tfidf_dataset(
    f'{BIGCN_DIR}/data/Twitter15/data.TD_RvNN.vol_5000.txt',
    t15_labels)

print("\nLoading Twitter16...")
t16_data, t16_y = load_tfidf_dataset(
    f'{BIGCN_DIR}/data/Twitter16/data.TD_RvNN.vol_5000.txt',
    t16_labels)

Loading Twitter15...


building graphs: 100%|██████████| 3098/3098 [00:01<00:00, 1977.19it/s]


Built 1470 graphs, skipped 1628

Loading Twitter16...


building graphs: 100%|██████████| 3098/3098 [00:00<00:00, 3956.22it/s]


Built 808 graphs, skipped 2290


In [4]:
# diagnostic — check what's in the file
with open(f'{BIGCN_DIR}/data/Twitter15/data.TD_RvNN.vol_5000.txt') as f:
    lines = f.readlines()

print(f"Total lines: {len(lines)}")
print(f"First 3 lines:")
for l in lines[:3]:
    print(repr(l[:100]))

# check how many unique root_ids
root_ids = set()
for l in lines:
    parts = l.strip().split('\t')
    if len(parts) >= 2:
        root_ids.add(parts[0].strip())
print(f"Unique root_ids: {len(root_ids)}")

# check label coverage
print(f"Labels available: {len(t15_labels)}")
print(f"Root IDs in data with labels: {len(root_ids & set(t15_labels.keys()))}")
print(f"Root IDs in data without labels: {len(root_ids - set(t15_labels.keys()))}")

Total lines: 161038
First 3 lines:
'656955120626880512\tNone\t1\t2\t9\t1:1 3:1 164:1 5:1 2282:1 11:1 431:1 473:1 729:1\n'
'656955120626880512\t1\t2\t2\t9\t0:2\n'
'624298742162845696\tNone\t3\t72\t26\t1:1 34:1 3:1 71:1 9:1 202:1 11:1 140:1 12:1 624:1 124:1 1266:1 692:1'
Unique root_ids: 3098
Labels available: 1490
Root IDs in data with labels: 1490
Root IDs in data without labels: 1608


In [5]:
class TDrumorGCN(nn.Module):
    def __init__(self, in_feats, hid_feats, out_feats):
        super().__init__()
        self.conv1 = GCNConv(in_feats, hid_feats)
        self.conv2 = GCNConv(hid_feats + in_feats, out_feats)

    def forward(self, data):
        x, edge_index = data.x, data.edge_index
        x1         = copy.copy(x.float())
        x          = self.conv1(x, edge_index)
        x2         = copy.copy(x)

        root_index  = data.root_index
        batch_size  = int(data.batch.max()) + 1
        root_extend = torch.zeros(len(data.batch),
                                   x1.size(1)).to(x.device)
        for b in range(batch_size):
            mask = (data.batch == b)
            root_extend[mask] = x1[root_index[b]]

        x = torch.cat((x, root_extend), dim=1)
        x = F.relu(x)
        x = F.dropout(x, p=0.2, training=self.training)
        x = self.conv2(x, edge_index)
        x = F.relu(x)

        root_extend2 = torch.zeros(len(data.batch),
                                    x2.size(1)).to(x.device)
        for b in range(batch_size):
            mask = (data.batch == b)
            root_extend2[mask] = x2[root_index[b]]

        x = torch.cat((x, root_extend2), dim=1)
        x = scatter_mean(x, data.batch, dim=0)
        return x

class BUrumorGCN(nn.Module):
    def __init__(self, in_feats, hid_feats, out_feats):
        super().__init__()
        self.conv1 = GCNConv(in_feats, hid_feats)
        self.conv2 = GCNConv(hid_feats + in_feats, out_feats)

    def forward(self, data):
        x, edge_index = data.x, data.BU_edge_index
        x1         = copy.copy(x.float())
        x          = self.conv1(x, edge_index)
        x2         = copy.copy(x)

        root_index  = data.root_index
        batch_size  = int(data.batch.max()) + 1
        root_extend = torch.zeros(len(data.batch),
                                   x1.size(1)).to(x.device)
        for b in range(batch_size):
            mask = (data.batch == b)
            root_extend[mask] = x1[root_index[b]]

        x = torch.cat((x, root_extend), dim=1)
        x = F.relu(x)
        x = F.dropout(x, p=0.2, training=self.training)
        x = self.conv2(x, edge_index)
        x = F.relu(x)

        root_extend2 = torch.zeros(len(data.batch),
                                    x2.size(1)).to(x.device)
        for b in range(batch_size):
            mask = (data.batch == b)
            root_extend2[mask] = x2[root_index[b]]

        x = torch.cat((x, root_extend2), dim=1)
        x = scatter_mean(x, data.batch, dim=0)
        return x

class BiGCN(nn.Module):
    def __init__(self, in_feats, hid_feats, out_feats, num_classes=4):
        super().__init__()
        self.td_gcn = TDrumorGCN(in_feats, hid_feats, out_feats)
        self.bu_gcn = BUrumorGCN(in_feats, hid_feats, out_feats)
        self.fc     = nn.Linear((out_feats + hid_feats) * 2, num_classes)

    def forward(self, data):
        td_x = self.td_gcn(data)
        bu_x = self.bu_gcn(data)
        x    = torch.cat((td_x, bu_x), dim=1)
        x    = self.fc(x)
        x    = F.log_softmax(x, dim=1)
        return x

In [6]:
def train_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0
    for data in loader:
        data = data.to(device)
        optimizer.zero_grad()
        out  = model(data)
        loss = criterion(out, data.y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=2.0)
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)

@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    all_preds  = []
    all_labels = []
    for data in loader:
        data  = data.to(device)
        out   = model(data)
        preds = out.argmax(dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(data.y.cpu().numpy())
    return np.array(all_preds), np.array(all_labels)

def run_cv(data_list, labels, dataset_name, n_splits=5,
           hid_feats=64, out_feats=64,
           epochs=200, lr=0.0005, batch_size=128):

    cv        = StratifiedKFold(n_splits=n_splits, shuffle=True,
                                random_state=42)
    all_preds = np.zeros(len(labels), dtype=int)

    for fold, (train_idx, val_idx) in enumerate(
            cv.split(data_list, labels)):
        print(f"  Fold {fold+1}/{n_splits}", end=' ')

        train_data   = [data_list[i] for i in train_idx]
        val_data     = [data_list[i] for i in val_idx]
        train_loader = DataLoader(train_data, batch_size=batch_size,
                                  shuffle=True)
        val_loader   = DataLoader(val_data,   batch_size=batch_size,
                                  shuffle=False)

        model     = BiGCN(in_feats=5000, hid_feats=hid_feats,
                          out_feats=out_feats, num_classes=4).to(device)

        # separate LR for BU — faithful to original
        BU_params  = list(map(id, model.bu_gcn.conv1.parameters()))
        BU_params += list(map(id, model.bu_gcn.conv2.parameters()))
        base_params = filter(lambda p: id(p) not in BU_params,
                             model.parameters())
        optimizer = Adam([
            {'params': base_params},
            {'params': model.bu_gcn.conv1.parameters(), 'lr': lr/5},
            {'params': model.bu_gcn.conv2.parameters(), 'lr': lr/5},
        ], lr=lr, weight_decay=1e-4)

        criterion    = torch.nn.NLLLoss()
        best_f1      = 0
        patience     = 10
        patience_ctr = 0
        best_preds   = None

        for epoch in range(epochs):
            loss = train_epoch(model, train_loader, optimizer, criterion)
            preds, labs = evaluate(model, val_loader)
            val_f1 = f1_score(labs, preds, average='macro')

            if val_f1 > best_f1:
                best_f1      = val_f1
                best_preds   = preds.copy()
                patience_ctr = 0
            else:
                patience_ctr += 1

            if (epoch + 1) % 10 == 0:
                print(f"\n    Epoch {epoch+1} | Loss: {loss:.4f} "
                      f"| Val F1: {val_f1:.4f}", end=' ')

            if patience_ctr >= patience:
                print(f"\n    Early stop at epoch {epoch+1}")
                break

        all_preds[val_idx] = best_preds if best_preds is not None \
                             else preds
        fold_f1  = f1_score(labels[val_idx], all_preds[val_idx],
                            average='macro')
        fold_acc = accuracy_score(labels[val_idx], all_preds[val_idx])
        print(f"| Best F1: {fold_f1:.4f} | Acc: {fold_acc:.4f}")

    macro_f1 = f1_score(labels, all_preds, average='macro')
    accuracy = accuracy_score(labels, all_preds)
    print(f"\n[{dataset_name}] Macro-F1: {macro_f1:.4f} | "
          f"Accuracy: {accuracy:.4f}")
    print(classification_report(labels, all_preds,
                                 target_names=['false', 'true',
                                               'unverified', 'non-rumor']))
    return all_preds, macro_f1, accuracy

In [7]:
sample      = [d for d in t15_data[:50]]
loader_test = DataLoader(sample, batch_size=8)
model_test  = BiGCN(in_feats=5000, hid_feats=64,
                    out_feats=64, num_classes=4).to(device)
batch       = next(iter(loader_test))
batch       = batch.to(device)
out         = model_test(batch)
print(f"Output shape: {out.shape}")  # should be (8, 4)
print("Sanity check passed")

Output shape: torch.Size([8, 4])
Sanity check passed


In [8]:
# run 3 iterations with different seeds and average
iteration_results = []

for seed in [42, 123, 456]:
    from sklearn.model_selection import StratifiedKFold
    # temporarily override random state
    t15_preds_iter, t15_f1_iter, t15_acc_iter = run_cv(
        t15_data, t15_y, dataset_name=f'T15_seed{seed}',
        # pass seed somehow
    )
    iteration_results.append(t15_acc_iter)

print(f"Mean accuracy: {np.mean(iteration_results):.4f}")
print(f"Std: {np.std(iteration_results):.4f}")

  Fold 1/5 
    Epoch 10 | Loss: 0.4098 | Val F1: 0.7860 
    Epoch 20 | Loss: 0.0570 | Val F1: 0.8206 
    Early stop at epoch 27
| Best F1: 0.8236 | Acc: 0.8231
  Fold 2/5 
    Epoch 10 | Loss: 0.4187 | Val F1: 0.8318 
    Epoch 20 | Loss: 0.0563 | Val F1: 0.8180 
    Early stop at epoch 20
| Best F1: 0.8318 | Acc: 0.8299
  Fold 3/5 
    Epoch 10 | Loss: 0.3974 | Val F1: 0.7604 
    Epoch 20 | Loss: 0.0525 | Val F1: 0.7831 
    Early stop at epoch 25
| Best F1: 0.7900 | Acc: 0.7891
  Fold 4/5 
    Epoch 10 | Loss: 0.4131 | Val F1: 0.7931 
    Epoch 20 | Loss: 0.0561 | Val F1: 0.8132 
    Early stop at epoch 24
| Best F1: 0.8173 | Acc: 0.8163
  Fold 5/5 
    Epoch 10 | Loss: 0.4130 | Val F1: 0.7872 
    Epoch 20 | Loss: 0.0582 | Val F1: 0.7943 
    Early stop at epoch 28
| Best F1: 0.8080 | Acc: 0.8061

[T15_seed42] Macro-F1: 0.8141 | Accuracy: 0.8129
              precision    recall  f1-score   support

       false       0.84      0.78      0.81       362
        true       0.91   

In [7]:
print("="*60)
print("TWITTER15 — TF-IDF BiGCN (full cascade)")
print("="*60)
t15_preds, t15_f1, t15_acc = run_cv(
    t15_data, t15_y, dataset_name='T15'
)

print("\n" + "="*60)
print("TWITTER16 — TF-IDF BiGCN (full cascade)")
print("="*60)
t16_preds, t16_f1, t16_acc = run_cv(
    t16_data, t16_y, dataset_name='T16'
)

TWITTER15 — TF-IDF BiGCN (full cascade)
  Fold 1/5 
    Epoch 10 | Loss: 0.4288 | Val F1: 0.7687 
    Epoch 20 | Loss: 0.0624 | Val F1: 0.8060 
    Epoch 30 | Loss: 0.0203 | Val F1: 0.8098 
    Epoch 40 | Loss: 0.0110 | Val F1: 0.8029 
    Early stop at epoch 42
| Best F1: 0.8131 | Acc: 0.8129
  Fold 2/5 
    Epoch 10 | Loss: 0.4174 | Val F1: 0.8174 
    Epoch 20 | Loss: 0.0579 | Val F1: 0.8064 
    Early stop at epoch 23
| Best F1: 0.8300 | Acc: 0.8299
  Fold 3/5 
    Epoch 10 | Loss: 0.4030 | Val F1: 0.7633 
    Epoch 20 | Loss: 0.0523 | Val F1: 0.7816 
    Early stop at epoch 27
| Best F1: 0.7875 | Acc: 0.7857
  Fold 4/5 
    Epoch 10 | Loss: 0.4261 | Val F1: 0.7794 
    Epoch 20 | Loss: 0.0562 | Val F1: 0.8174 
    Epoch 30 | Loss: 0.0193 | Val F1: 0.7961 
    Early stop at epoch 30
| Best F1: 0.8174 | Acc: 0.8163
  Fold 5/5 
    Epoch 10 | Loss: 0.4060 | Val F1: 0.7884 
    Epoch 20 | Loss: 0.0591 | Val F1: 0.8011 
    Early stop at epoch 27
| Best F1: 0.8084 | Acc: 0.8061

[T15] 

In [8]:
results = pd.DataFrame([
    {'dataset': 'T15', 'macro_f1': t15_f1, 'accuracy': t15_acc},
    {'dataset': 'T16', 'macro_f1': t16_f1, 'accuracy': t16_acc},
])
results.to_csv(f'{OUTPUTS}/tfidf_results.csv', index=False)
print(results.to_string())

print("\nBian et al. reported:")
print("T15: Accuracy=0.886, F_NR=0.930, F_FR=0.891, F_TR=0.860, F_UR=0.864")
print("T16: Accuracy=0.880, F_NR=0.937, F_FR=0.847, F_TR=0.869, F_UR=0.865")

  dataset  macro_f1  accuracy
0     T15  0.811289  0.810204
1     T16  0.832417  0.832921

Bian et al. reported:
T15: Accuracy=0.886, F_NR=0.930, F_FR=0.891, F_TR=0.860, F_UR=0.864
T16: Accuracy=0.880, F_NR=0.937, F_FR=0.847, F_TR=0.869, F_UR=0.865
